In [2]:
import numpy as np
import xarray as xr
import pandas as pd
import re
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from metpy.plots import colortables
import os

In [3]:
slgt = False
if slgt:
    slgt_str = '_slgt'
else:
    slgt_str = ''

In [60]:
# split days into positive/negative/near zero bias (for one of 12 bias types)--can be done with just targets!
targets = xr.open_dataset(f'data/processed_data/train_targets{slgt_str}.nc')

In [78]:
sd_split = .431

variables = targets.variable.values

# Preallocate the output variable directly (same shape/coords as desired)
day_ds = xr.Dataset(
    {
        "bias_sign": (
            ("hazard", "variable", "time"),
            np.empty((len(targets.hazard), len(variables), len(targets.time)), dtype=object),
        )
    },
    coords={
        "hazard": targets.hazard,
        "variable": variables,
        "time": targets.time,
    },
)

# Fill bias_sign for each variable
for variable in variables:
    vals = targets[variable]  # shape: (hazard, time)
    day_ds["bias_sign"].loc[dict(variable=variable)] = xr.where(
        vals > sd_split,
        "pos",
        xr.where(vals < -sd_split, "neg", "zero"),
    )


In [79]:
day_ds.to_netcdf(f"data/processed_data/day_splits{slgt_str}.nc")

Making Plots

In [154]:
day_ds = xr.open_dataset(f"data/processed_data/day_splits{slgt_str}.nc")

In [155]:
# plot composite (uncentered) weather variable and outlook of given set of days function
# plot composite (centered) weather variable and outlook of given set of days function
def plot_composites(outlooks, weather, days, var_to_plot, bias_variable, bias_sign,
                    hazard=None, centered=False, level=None, tod=0, save=True, show=False):
    """
    Plot composites of either an outlook probability field or a weather variable.
    """
    outlooks = outlooks.sel(time=days)
    weather = weather.sel(day=days)

    # --- 1. Outlook case ---
    if var_to_plot == "prob":
        if hazard is None:
            raise ValueError("Must specify a hazard when plotting outlook probabilities.")
        sub = outlooks.sel(hazard=hazard)
        comp = sub["prob"].mean(dim="time")

        lats = outlooks["lat"]
        lons = outlooks["lon"]
        title = f"Composite {var_to_plot} field when {hazard} {bias_variable} is {bias_sign}"

    # --- 2. Weather case ---
    elif var_to_plot in weather.data_vars:
        sub = weather[var_to_plot]

        # Handle level (default 500 hPa if applicable)
        if "level" in sub.dims:
            use_level = 500 if level is None else level
            if use_level in sub["level"]:
                sub = sub.sel(level=use_level)
                lev_text = f", {use_level} hPa"
            else:
                raise ValueError(f"Requested level {use_level} not found in variable {var_to_plot}.")
        else:
            lev_text = ""

        # Handle tod (always select one, default 0)
        if "tod" in sub.dims:
            sub = sub.sel(tod=tod)

        # Average over days
        comp = sub.mean(dim="day")

        lats = weather.get("latitude", weather.get("lat"))
        lons = weather.get("longitude", weather.get("lon"))
        title = f"Composite {var_to_plot} anomaly field when {hazard} {bias_variable} is {bias_sign}"

    else:
        raise KeyError(f"Variable {var_to_plot} not found in outlooks or weather datasets.")

    # --- 3. Plot composite map ---
    if centered:
        # === Centered case ===
        fig, ax = plt.subplots(figsize=(7, 6))
        title = "Centered " + title

        # Adjust color scale
        if var_to_plot == "prob":
            cmap = colortables.get_colortable("NWSReflectivity")
            vmin, vmax = 0, .45
            if hazard == "Tornado":
                vmax = .1
        else:
            cmap = plt.get_cmap("coolwarm")
            vmin, vmax = -1, 1

        if var_to_plot == "prob": 
            xs = outlooks['x']
            ys = outlooks['y']
            scale = 80
        else:
            xs = weather['longitude']
            ys = weather['latitude']
            scale = 1

        # Plot flat in grid space
        pcm = ax.pcolormesh(
            xs*scale, ys*scale,
            comp,
            cmap=cmap,
            vmin=vmin,
            vmax=vmax
        )
        
        if var_to_plot == "prob":
            x_ticks = np.arange(-1000, 1001, 250)
            y_ticks = np.arange(-1000, 1001, 250)
        else:
            # Compute symmetric range around zero
            x_range = int(np.ceil(max(abs(xs.min()), abs(xs.max()))))
            y_range = int(np.ceil(max(abs(ys.min()), abs(ys.max()))))

            # Force ranges to be multiples of 10 so that 0 is always centered
            x_range = int(np.ceil(x_range / 10)) * 10
            y_range = int(np.ceil(y_range / 10)) * 10

            # Create ticks every 10 degrees centered at 0
            x_ticks = np.arange(-x_range, x_range + 1, 10)
            y_ticks = np.arange(-y_range, y_range + 1, 10)
        for xt in x_ticks:
            ax.axvline(xt, color='gray', lw=0.3, alpha=0.4)
        for yt in y_ticks:
            ax.axhline(yt, color='gray', lw=0.3, alpha=0.4)
        
        # todo: relabel axes ()
        if var_to_plot == "prob":
            ax.set_xlabel("x (km from outlook center)")
            ax.set_ylabel("y (km from outlook center)")
            ax.set_xlim(-1000, 1000)
            ax.set_ylim(-1000, 1000)
        else:
            ax.set_xlabel("Relative Longitude (degrees from center)")
            ax.set_ylabel("Relative Latitude (degrees from center)")
        ax.set_title(title)
        plt.colorbar(pcm, ax=ax, orientation="horizontal", pad=0.05, label=var_to_plot)

        if save:
            outdir = f'figs/centered_composites/{var_to_plot}/'
            os.makedirs(outdir, exist_ok=True)
            plt.savefig(
                f'{outdir}{hazard}_{bias_variable}_{bias_sign}_composite.png',
                dpi=300,
                bbox_inches='tight'
            )

    else:
        # === Geographic case ===
        fig, ax = plt.subplots(
            figsize=(9, 6),
            subplot_kw={"projection": ccrs.LambertConformal(central_longitude=-95, central_latitude=35)}
        )

        ax.set_extent([-125, -67, 25, 50], crs=ccrs.PlateCarree())
        ax.add_feature(cfeature.STATES, linewidth=0.7)
        ax.add_feature(cfeature.COASTLINE, linewidth=0.7)
        ax.add_feature(cfeature.BORDERS, linewidth=0.5)

        if var_to_plot == "prob":
            cmap = colortables.get_colortable("NWSReflectivity")
            vmin, vmax = 0, 0.15
            if hazard == "Tornado":
                vmax = 0.05
        else:
            cmap = plt.get_cmap("coolwarm")
            vmin, vmax = -1, 1

        pcm = ax.pcolormesh(
            lons,
            lats,
            comp,
            vmin=vmin,
            vmax=vmax,
            cmap=cmap,
            transform=ccrs.PlateCarree()
        )

        plt.colorbar(pcm, ax=ax, orientation="horizontal", pad=0.05, label=var_to_plot)
        ax.set_title(title)

        if save:
            outdir = f'figs/composites/{var_to_plot}/'
            os.makedirs(outdir, exist_ok=True)
            plt.savefig(
                f'{outdir}{hazard}_{bias_variable}_{bias_sign}_composite.png',
                dpi=300,
                bbox_inches='tight'
            )

    if show:
        plt.show()
    plt.close()

    # TODO: handle wind barbs?. And c

In [156]:
outlooks = xr.open_dataset('data/raw_data/outlooks.nc')
weather = xr.open_zarr('data/processed_data/train_inputs_small.zarr').load()

In [157]:
def compute_outlook_centers(outlooks, hazard):
    """
    For each day in outlooks, find the mean (x,y) index and mean (lat,lon)
    of the gridpoints that equal the day's max probability for `hazard`.
    """
    sub = outlooks.sel(hazard=hazard)
    times = sub.time.values

    x_centers, y_centers, lat_centers, lon_centers = [], [], [], []

    for t in times:
        p = sub["prob"].sel(time=t)
        maxval = p.max().item()
        mask_vals = (p == maxval).values
        y_inds, x_inds = np.where(mask_vals)

        x_mean = sub["x"].values[x_inds].mean()
        y_mean = sub["y"].values[y_inds].mean()
        lat_mean = sub["lat"].values[y_inds, x_inds].mean()
        lon_mean = sub["lon"].values[y_inds, x_inds].mean()

        x_centers.append(x_mean)
        y_centers.append(y_mean)
        lat_centers.append(lat_mean)
        lon_centers.append(lon_mean)

    centers = xr.Dataset(
        {
            "x_center": ("time", np.array(x_centers)),
            "y_center": ("time", np.array(y_centers)),
            "lat_center": ("time", np.array(lat_centers)),
            "lon_center": ("time", np.array(lon_centers)),
        },
        coords={"time": times},
    )
    return centers


def center_datasets(outlooks, weather, hazard='All Hazard', fill_value=0):
    """
    Recenter outlooks and weather per-day based on outlook max-prob centroids.

    Each day's outlook and weather are regridded to a common relative
    coordinate system (x_rel, y_rel) or (rel_lat, rel_lon).
    """
    # --- compute centers ---
    centers = compute_outlook_centers(outlooks, hazard)

    outlook_slices = []
    weather_slices = []

    out_x = outlooks["x"].values
    out_y = outlooks["y"].values
    wx_lat = weather["latitude"].values
    wx_lon = weather["longitude"].values

    for i, t in enumerate(centers.time.values):
        cx = centers["x_center"].isel(time=i).item() + .001 # to avoid weird rounding??
        cy = centers["y_center"].isel(time=i).item() + .001 # to avoid weird rounding??
        latc = centers["lat_center"].isel(time=i).item()
        lonc = centers["lon_center"].isel(time=i).item()

        # --- Outlook centering ---
        # Compute *rounded* relative coordinates
        x_rel = np.round(out_x - cx).astype(int)
        y_rel = np.round(out_y - cy).astype(int)

        day_out = outlooks.sel(time=t).copy(deep=False)

        # Replace coordinate names and values so x_rel/y_rel become the new spatial coords
        day_out = day_out.assign_coords({"x": x_rel, "y": y_rel})
        day_out = day_out.expand_dims({"time": [np.datetime64(t)]})
        outlook_slices.append(day_out)

        # --- Weather centering ---
        # Compute native grid spacing (assuming regular grid)
        dy = float(np.median(np.diff(wx_lat)))
        dx = float(np.median(np.diff(wx_lon)))

        # Round to the nearest gridpoint multiple of that spacing
        rel_lat = np.round((wx_lat - latc) / dy) * dy
        rel_lon = np.round((wx_lon - lonc) / dx) * dx

        day_wx = weather.sel(day=t).copy(deep=False)
        day_wx = day_wx.assign_coords({
            "latitude": rel_lat,
            "longitude": rel_lon - 360
        })
        day_wx = day_wx.expand_dims({"day": [np.datetime64(t)]})
        weather_slices.append(day_wx)

    # --- Combine along time/day, aligning on the new relative coords ---
    centered_outlooks = xr.concat(outlook_slices, dim="time", join="outer").fillna(fill_value)
    centered_weather = xr.concat(weather_slices, dim="day", join="outer").fillna(fill_value)

    # Rename to make the relative nature explicit
    #centered_outlooks = centered_outlooks.rename({"x": "x_rel", "y": "y_rel"})
    #centered_weather = centered_weather.rename({"latitude": "rel_lat", "longitude": "rel_lon"})

    return centered_outlooks, centered_weather, centers

In [158]:
centered_outlooks, centered_weather, centers = center_datasets(outlooks, weather, hazard='All Hazard')

C:\Users\miles\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.10_qbz5n2kfra8p0\LocalCache\local-packages\Python310\site-packages\xarray\core\concat.py:532: FutureWarning: unique with argument that is not not a Series, Index, ExtensionArray, or np.ndarray is deprecated and will raise in a future version.
  common_dims = tuple(pd.unique([d for v in vars for d in v.dims]))


In [159]:
for hazard in day_ds.hazard.values:
    for variable in day_ds.variable.values:
        for bias_sign in ['pos', 'neg', 'zero']:
            days = day_ds['time'][(day_ds.bias_sign.sel(hazard=hazard, variable=variable) == bias_sign).values].values
            
            if len(days) > 0:
                #print(days)
                #if hazard == "Wind" and variable == "north_shift":
                print(f'Hazard: {hazard}, Variable: {variable}, Bias Sign: {bias_sign}, Number of Days: {len(days)}')
                for var in list(weather.data_vars) + ['prob']:
                    plot_composites(outlooks, weather, days, var, variable, bias_sign, hazard, centered=False) 
                    plot_composites(centered_outlooks, centered_weather, days, var, variable, bias_sign, hazard, centered=True) 

Hazard: All Hazard, Variable: bias, Bias Sign: pos, Number of Days: 122
Hazard: All Hazard, Variable: bias, Bias Sign: neg, Number of Days: 90
Hazard: All Hazard, Variable: bias, Bias Sign: zero, Number of Days: 120
Hazard: All Hazard, Variable: east_shift, Bias Sign: pos, Number of Days: 98
Hazard: All Hazard, Variable: east_shift, Bias Sign: neg, Number of Days: 104
Hazard: All Hazard, Variable: east_shift, Bias Sign: zero, Number of Days: 130
Hazard: All Hazard, Variable: north_shift, Bias Sign: pos, Number of Days: 102
Hazard: All Hazard, Variable: north_shift, Bias Sign: neg, Number of Days: 107
Hazard: All Hazard, Variable: north_shift, Bias Sign: zero, Number of Days: 123
Hazard: Wind, Variable: bias, Bias Sign: pos, Number of Days: 106
Hazard: Wind, Variable: bias, Bias Sign: neg, Number of Days: 90
Hazard: Wind, Variable: bias, Bias Sign: zero, Number of Days: 136
Hazard: Wind, Variable: east_shift, Bias Sign: pos, Number of Days: 105
Hazard: Wind, Variable: east_shift, Bias S

In [141]:
dy = float(np.median(np.diff(weather.latitude.values)))
dx = float(np.median(np.diff(weather.longitude.values)))
print(dy, dx)

-1.0 1.0


In [94]:
list(weather.data_vars) + ['a']

['10m_u_component_of_wind',
 '10m_v_component_of_wind',
 '2m_dewpoint_temperature',
 '2m_temperature',
 'geopotential',
 'geopotential_at_surface',
 'potential_vorticity',
 'specific_humidity',
 'temperature',
 'toa_incident_solar_radiation',
 'u_component_of_wind',
 'v_component_of_wind',
 'vertical_velocity',
 'a']

In [65]:
centered_outlooks

<xarray.Dataset>
Dimensions:  (time: 332, y_rel: 90, x_rel: 129)
Coordinates:
  * x_rel    (x_rel) int32 -74 -73 -72 -71 -70 -69 -68 ... 48 49 50 51 52 53 54
  * y_rel    (y_rel) int32 -42 -41 -40 -39 -38 -37 -36 ... 41 42 43 44 45 46 47
  * time     (time) datetime64[ns] 2002-04-07 2002-04-13 ... 2019-05-20
    hazard   <U10 'All Hazard'
Data variables:
    lat      (time, y_rel, x_rel) float64 0.0 0.0 0.0 0.0 ... 0.0 0.0 0.0 0.0
    lon      (time, y_rel, x_rel) float64 0.0 0.0 0.0 0.0 ... 0.0 0.0 0.0 0.0
    prob     (time, y_rel, x_rel) float64 0.0 0.0 0.0 0.0 ... 0.0 0.0 0.0 0.0
Attributes:
    description:  outlook as a percentage as a function of date, lat/lon, and...
    grid:         80-km NCEP 211

In [ ]:
# for a specific model, for a specific target variable, break days somehow--days where model predicts positive, negative, and near zero. Or where model error is +/-/0? Then cross with true biases, and composite each of those categories.

Only run once, don't need to again:

In [90]:
# plot composites and look at examples from 
outlooks = xr.open_dataset('data/raw_data/grid_outlooks.nc')
outlooks = outlooks.assign_coords(time=pd.to_datetime(outlooks.time.astype(str), format='%Y%m%d%H%M'))

# Select Day 1 outlooks (including hazard types)
outlooks = outlooks.sel(outlook=[o for o in outlooks.outlook.values if o.startswith('Day 1')])

# Select only times in known list
outlooks = outlooks.sel(time=outlooks.time.isin(day_ds['time']))

def map_outlook_to_hazard(o):
    """Map Day 1 outlook strings to clean hazard names."""
    if o == 'Day 1':
        return 'All Hazard'
    elif re.search(r'Wind', o):
        return 'Wind'
    elif re.search(r'Hail', o):
        return 'Hail'
    elif re.search(r'Tornado', o):
        return 'Tornado'
    else:
        return None  # unexpected case

# Apply mapping
hazards = [map_outlook_to_hazard(o) for o in outlooks.outlook.values]

# Assign new coordinate
outlooks = outlooks.assign_coords(hazard=('outlook', hazards))

# Drop the old 'outlook' coordinate and rename
outlooks = outlooks.swap_dims({'outlook': 'hazard'}).drop_vars('outlook')
outlooks.to_netcdf("data/raw_data/outlooks.nc")